# P7513 MERSCOPE table vs table_MOSAIK_proseg clustering

This notebook compares P7513 MERSCOPE clustering when the only intended input difference is the SpatialData table: `table` versus `table_MOSAIK_proseg`.

The clustering run uses the current MerXen `ClusteringSquidpyConfig` defaults, including hierarchical clustering, broad Leiden resolution, control-feature filtering, PCA/neighbors/UMAP settings, and GPU request/fallback behavior from the current code.

## Environment

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import anndata as ad
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import scanpy as sc
import spatialdata as sd
from scipy import sparse
from sklearn.metrics import adjusted_rand_score

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "merxen").exists():
    REPO_ROOT = Path("/home/becalia/code/MerXen")

src_path = REPO_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from merxen.analysis.clustering_squidpy import (
    BROAD_CLASS_KEY,
    BROAD_CLUSTER_KEY,
    HIERARCHICAL_CLUSTER_KEY,
    run_clustering_squidpy,
)
from merxen.config import ClusteringSquidpyConfig, ClusteringSquidpySampleConfig
from merxen.plotting import save_figure

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)
print(f"Repo root: {REPO_ROOT}")

## Inputs

`shape_key` is left as `None` in both test samples so the current loader chooses the matching spatial layer from each table's metadata. That keeps the clustering parameters current-code exact while changing only the selected table key. Expression clustering does not use the spatial coordinates, but the chosen shape affects spatial plots and shape-derived QC fields.

In [ ]:
PAIR_ID = "P7513"
PLATFORM = "MERSCOPE"
PLATFORM_DIR = PLATFORM.lower()

ZARR_CANDIDATES = [
    Path("/media/mathieubo/SSD2/MerXen/results") / PAIR_ID / PLATFORM_DIR / "latest" / "latest_spatialdata.zarr",
    REPO_ROOT / "results" / PAIR_ID / PLATFORM_DIR / "latest" / "latest_spatialdata.zarr",
]
ZARR_PATH = next((path for path in ZARR_CANDIDATES if path.exists()), ZARR_CANDIDATES[0])
if not ZARR_PATH.exists():
    raise FileNotFoundError("Could not find P7513 MERSCOPE SpatialData zarr. Checked: " + ", ".join(str(path) for path in ZARR_CANDIDATES))

OUTPUT_DIR = (
    REPO_ROOT
    / "results"
    / PAIR_ID
    / "merscope_table_vs_MOSAIK_proseg_current_clustering"
)

TABLE_RUNS = [
    {
        "label": "table",
        "sample_id": f"{PAIR_ID}_{PLATFORM}_table",
        "table_key": "table",
        "shape_key": None,
    },
    {
        "label": "table_MOSAIK_proseg",
        "sample_id": f"{PAIR_ID}_{PLATFORM}_table_MOSAIK_proseg",
        "table_key": "table_MOSAIK_proseg",
        "shape_key": None,
    },
]

print(f"SpatialData zarr: {ZARR_PATH}")
print(f"Output dir:       {OUTPUT_DIR}")
for run in TABLE_RUNS:
    print(run)

## Preflight: compare the raw tables

This cell checks whether the two tables have the same cells, the same genes, and the same count matrix after aligning gene order. It is intentionally run before clustering so table-level differences are visible without any Scanpy preprocessing.

In [ ]:
def _matrix_sum(x) -> float:
    return float(x.sum() if sparse.issparse(x) else np.asarray(x).sum())


def _matrix_nnz(x) -> int:
    return int(x.nnz if sparse.issparse(x) else np.count_nonzero(x))


def _diff_stats(left: ad.AnnData, right: ad.AnnData) -> dict[str, object]:
    stats: dict[str, object] = {
        "same_shape": left.shape == right.shape,
        "same_obs_names": left.obs_names.equals(right.obs_names),
        "same_var_names_order": left.var_names.equals(right.var_names),
        "same_var_name_set": set(left.var_names) == set(right.var_names),
    }
    if not (stats["same_shape"] and stats["same_obs_names"] and stats["same_var_name_set"]):
        return stats

    right_aligned = right[:, left.var_names]
    diff = left.X - right_aligned.X
    if sparse.issparse(diff):
        stats["matrix_diff_nnz"] = int(diff.nnz)
        stats["matrix_diff_abs_sum"] = float(abs(diff).sum())
    else:
        diff_array = np.asarray(diff)
        stats["matrix_diff_nnz"] = int(np.count_nonzero(diff_array))
        stats["matrix_diff_abs_sum"] = float(np.abs(diff_array).sum())

    left_counts = np.asarray(left.X.sum(axis=1)).ravel()
    right_counts = np.asarray(right_aligned.X.sum(axis=1)).ravel()
    delta = right_counts - left_counts
    stats.update(
        {
            "right_minus_left_count_sum": float(delta.sum()),
            "right_minus_left_count_median": float(np.median(delta)),
            "right_minus_left_count_p01": float(np.percentile(delta, 1)),
            "right_minus_left_count_p99": float(np.percentile(delta, 99)),
            "cells_increased": int(np.sum(delta > 0)),
            "cells_decreased": int(np.sum(delta < 0)),
            "cells_unchanged": int(np.sum(delta == 0)),
        }
    )
    return stats


sdata = sd.read_zarr(ZARR_PATH)
print("tables:", list(sdata.tables.keys()))
print("images:", list(sdata.images.keys()))
print("shapes:", list(sdata.shapes.keys()))

table_rows = []
for run in TABLE_RUNS:
    table_key = run["table_key"]
    table = sdata.tables[table_key]
    attrs = dict(table.uns.get("spatialdata_attrs", {}))
    table_rows.append(
        {
            "label": run["label"],
            "table_key": table_key,
            "n_cells": table.n_obs,
            "n_features": table.n_vars,
            "count_sum": _matrix_sum(table.X),
            "matrix_nnz": _matrix_nnz(table.X),
            "region": attrs.get("region"),
            "region_key": attrs.get("region_key"),
            "instance_key": attrs.get("instance_key"),
        }
    )

table_summary = pd.DataFrame(table_rows)
display(table_summary)

diff_summary = _diff_stats(sdata.tables["table"], sdata.tables["table_MOSAIK_proseg"])
display(pd.Series(diff_summary, name="table_MOSAIK_proseg_vs_table"))

del sdata

## Current-code clustering config

The config below intentionally does not override clustering parameters. It uses the current defaults from `ClusteringSquidpyConfig`, with two MERSCOPE samples that differ only in `table_key`.

In [ ]:
samples = [
    ClusteringSquidpySampleConfig(
        sample_id=run["sample_id"],
        platform=PLATFORM,
        zarr_path=ZARR_PATH,
        segmentation="table_comparison",
        table_key=run["table_key"],
        shape_key=run["shape_key"],
    )
    for run in TABLE_RUNS
]

cfg = ClusteringSquidpyConfig(
    pair_id=f"{PAIR_ID}_{PLATFORM}_table_comparison",
    output_dir=OUTPUT_DIR,
    samples=samples,
)

config_summary = {
    "drop_control_features": cfg.drop_control_features,
    "min_counts": cfg.min_counts,
    "min_cells": cfg.min_cells,
    "normalize_target_sum": cfg.normalize_target_sum,
    "normalize_exclude_highly_expressed": cfg.normalize_exclude_highly_expressed,
    "normalize_max_fraction": cfg.normalize_max_fraction,
    "n_pcs": cfg.n_pcs,
    "n_neighbors": cfg.n_neighbors,
    "leiden_resolution": cfg.leiden_resolution,
    "umap_min_dist": cfg.umap_min_dist,
    "umap_spread": cfg.umap_spread,
    "random_seed": cfg.random_seed,
    "use_gpu": cfg.use_gpu,
    "hierarchical_enabled": cfg.hierarchical_enabled,
    "broad_leiden_resolution": cfg.broad_round.leiden_resolution,
    "subcluster_leiden_resolution": cfg.subcluster_round.leiden_resolution,
    "neuron_split_leiden_resolution": cfg.neuron_split_round.leiden_resolution,
    "neuron_subcluster_leiden_resolution": cfg.neuron_subcluster_round.leiden_resolution,
    "min_branch_cells": cfg.min_branch_cells,
}
display(pd.Series(config_summary, name="current_clustering_defaults"))
display(pd.DataFrame([sample.model_dump() for sample in samples]))

## Run clustering

This calls the same top-level function as the current pipeline. It will write h5ad files and production plots into `OUTPUT_DIR`. Expect this cell to take a while.

In [ ]:
RUN_CLUSTERING = True

if RUN_CLUSTERING:
    clustering_results = run_clustering_squidpy(cfg)
else:
    clustering_results = {}

clustering_results

## Load clustered outputs

In [ ]:
H5AD_PATHS = {
    run["label"]: OUTPUT_DIR / PLATFORM_DIR / f"{run['sample_id']}_clustered.h5ad"
    for run in TABLE_RUNS
}

for label, path in H5AD_PATHS.items():
    print(f"{label}: {path} ({'exists' if path.exists() else 'missing'})")

clustered = {label: ad.read_h5ad(path) for label, path in H5AD_PATHS.items()}
clustered

## Cluster and QC summary

In [ ]:
def _first_existing_params(adata_obj: ad.AnnData) -> dict[str, object]:
    for key in ["merxen_clustering_params_leiden_broad", "merxen_clustering_params"]:
        if key in adata_obj.uns:
            return dict(adata_obj.uns[key])
    return {}


def _cluster_count(adata_obj: ad.AnnData, key: str) -> int | None:
    if key not in adata_obj.obs:
        return None
    return int(adata_obj.obs[key].astype(str).nunique())


summary_rows = []
for label, adata_obj in clustered.items():
    meta = dict(adata_obj.uns.get("merxen_clustering_squidpy", {}))
    params = _first_existing_params(adata_obj)
    summary_rows.append(
        {
            "label": label,
            "table_key": meta.get("table_key"),
            "shape_key": meta.get("shape_key"),
            "n_cells_after_filtering": adata_obj.n_obs,
            "n_features_after_filtering": adata_obj.n_vars,
            "count_sum_layer_counts": _matrix_sum(adata_obj.layers["counts"]) if "counts" in adata_obj.layers else np.nan,
            "leiden_broad_clusters": _cluster_count(adata_obj, BROAD_CLUSTER_KEY),
            "broad_classes": _cluster_count(adata_obj, BROAD_CLASS_KEY),
            "hierarchical_clusters": _cluster_count(adata_obj, HIERARCHICAL_CLUSTER_KEY),
            "gpu_used": params.get("gpu_used"),
            "n_neighbors": params.get("n_neighbors"),
            "leiden_resolution": params.get("leiden_resolution"),
            "umap_min_dist": params.get("umap_min_dist"),
        }
    )

cluster_summary = pd.DataFrame(summary_rows)
display(cluster_summary)

for label, adata_obj in clustered.items():
    print(f"\n{label}")
    for key in [BROAD_CLUSTER_KEY, BROAD_CLASS_KEY, HIERARCHICAL_CLUSTER_KEY, "leiden"]:
        if key in adata_obj.obs:
            print(key)
            display(adata_obj.obs[key].astype(str).value_counts().head(20).to_frame("n_cells"))

## Same-cell agreement

This compares labels on cells retained in both clustered outputs. ARI is label-name agnostic, so it is useful for `leiden_broad` and `hierarchical_cluster`; direct equality is useful for broad class labels.

In [ ]:
left_label, right_label = [run["label"] for run in TABLE_RUNS]
left = clustered[left_label]
right = clustered[right_label]
common_cells = left.obs_names.intersection(right.obs_names)

agreement_rows = []
for key in [BROAD_CLUSTER_KEY, BROAD_CLASS_KEY, HIERARCHICAL_CLUSTER_KEY, "leiden"]:
    if key not in left.obs or key not in right.obs:
        continue
    left_values = left.obs.loc[common_cells, key].astype(str)
    right_values = right.obs.loc[common_cells, key].astype(str)
    agreement_rows.append(
        {
            "label_key": key,
            "n_common_cells": len(common_cells),
            "left_unique": left_values.nunique(),
            "right_unique": right_values.nunique(),
            "exact_match_fraction": float((left_values.to_numpy() == right_values.to_numpy()).mean()),
            "adjusted_rand_index": float(adjusted_rand_score(left_values, right_values)),
        }
    )

display(pd.DataFrame(agreement_rows))

## Side-by-side plots

These plotting helpers follow the spirit of `P7513_clustering_squidpy_interactive.ipynb`: explicit Matplotlib scatter panels, shared legends, and optional PNG/PDF saving via the repo `save_figure()` helper.

In [ ]:
def _category_order(values: pd.Series) -> list[str]:
    categories = list(pd.unique(values.astype(str)))
    try:
        return sorted(categories, key=lambda value: float(value))
    except ValueError:
        return sorted(categories)


def _palette_for_values(values: pd.Series) -> dict[str, object]:
    categories = _category_order(values)
    cmap = plt.get_cmap("tab20") if len(categories) <= 20 else plt.get_cmap("tab20b")
    return {category: cmap(index % cmap.N) for index, category in enumerate(categories)}


def _scatter_categorical(
    ax: plt.Axes,
    xy: np.ndarray,
    labels: pd.Series,
    *,
    point_size: float,
    alpha: float,
    title: str,
) -> None:
    label_text = labels.astype(str)
    palette = _palette_for_values(label_text)
    for category in _category_order(label_text):
        mask = label_text.to_numpy() == category
        ax.scatter(
            xy[mask, 0],
            xy[mask, 1],
            s=point_size,
            c=[palette[category]],
            alpha=alpha,
            linewidths=0,
            edgecolors="none",
            rasterized=True,
        )
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal", adjustable="datalim")


def plot_table_comparison(
    clustered_by_label: dict[str, ad.AnnData],
    *,
    color_key: str = BROAD_CLUSTER_KEY,
    embedding_key: str = "X_umap",
    point_size: float = 0.35,
    alpha: float = 0.7,
    figsize: tuple[float, float] = (12.0, 5.8),
) -> plt.Figure:
    n = len(clustered_by_label)
    fig, axes = plt.subplots(1, n, figsize=figsize, constrained_layout=True)
    if n == 1:
        axes = [axes]
    for ax, (label, adata_obj) in zip(axes, clustered_by_label.items(), strict=True):
        if color_key not in adata_obj.obs:
            raise KeyError(f"{label} is missing obs column {color_key!r}.")
        if embedding_key == "spatial":
            xy = np.asarray(adata_obj.obsm["spatial"])
        else:
            xy = np.asarray(adata_obj.obsm[embedding_key])
        _scatter_categorical(
            ax,
            xy,
            adata_obj.obs[color_key],
            point_size=point_size,
            alpha=alpha,
            title=f"{label}\n{embedding_key} colored by {color_key}",
        )
    return fig


for color_key in [BROAD_CLUSTER_KEY, BROAD_CLASS_KEY, HIERARCHICAL_CLUSTER_KEY]:
    if not all(color_key in adata_obj.obs for adata_obj in clustered.values()):
        continue
    fig = plot_table_comparison(clustered, color_key=color_key, embedding_key="X_umap")
    output_path = OUTPUT_DIR / f"{PAIR_ID}_{PLATFORM}_table_comparison_umap_{color_key}.png"
    save_figure(fig, output_path, dpi=220)
    print(f"Saved {output_path}")
    plt.show()

fig = plot_table_comparison(
    clustered,
    color_key=BROAD_CLASS_KEY if all(BROAD_CLASS_KEY in x.obs for x in clustered.values()) else "leiden",
    embedding_key="spatial",
    point_size=0.3,
    alpha=0.65,
)
output_path = OUTPUT_DIR / f"{PAIR_ID}_{PLATFORM}_table_comparison_spatial_broad_class.png"
save_figure(fig, output_path, dpi=220)
print(f"Saved {output_path}")
plt.show()

## Optional: inspect one table with old-style single-pass clustering

Leave this section untouched for the current-code comparison. It is here only as a convenient next step if you want to separate table effects from the hierarchical broad-clustering display effect later.

In [ ]:
# Example next step, not run by default:
# from merxen.analysis.clustering_squidpy import load_spatialdata_adata, run_scanpy_clustering
# adata_table = load_spatialdata_adata(ZARR_PATH, platform=PLATFORM, table_key="table")
# old_style = run_scanpy_clustering(
#     adata_table,
#     drop_control_features=cfg.drop_control_features,
#     min_counts=cfg.min_counts,
#     min_cells=cfg.min_cells,
#     normalize_target_sum=cfg.normalize_target_sum,
#     normalize_exclude_highly_expressed=cfg.normalize_exclude_highly_expressed,
#     normalize_max_fraction=cfg.normalize_max_fraction,
#     n_pcs=cfg.n_pcs,
#     n_neighbors=cfg.n_neighbors,
#     leiden_resolution=cfg.leiden_resolution,
#     umap_min_dist=cfg.umap_min_dist,
#     umap_spread=cfg.umap_spread,
#     random_seed=cfg.random_seed,
#     use_gpu=cfg.use_gpu,
# )